In [7]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 1b. Gráficos del sensor interior y variables exteriores
# DESCRIPCIÓN:
#   Carga datos del sensor interior (pasillo), ocupación y variables
#   exteriores (Tª, humedad, radiación solar). Genera scatter plots
#   por mes (dic-mar) correlacionando Tª, CO2, humedad y ocupación,
#   incluyendo relaciones interior-exterior.
#
# UBICACIÓN DEL SENSOR EN EL AULA 1.6:
#   - Corridor (Pasillo): Único sensor, junto a la pared, orientado al pasillo.
# ===================================================================

import pandas as pd
import os
import glob
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime, timedelta, time

# Directorio de salida para los gráficos
OUTPUT_DIR = 'plots_scatters_1.6'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# CARGAR DATOS DE OCUPACIÓN (archivo separado)
# ==========================================
print("\n" + "="*60)
print("CARGANDO DATOS DE OCUPACIÓN...")
print("="*60)

OCCUPANCY_FILE = '1.6 occupancy.xlsx'
df_occ = None

if os.path.exists(OCCUPANCY_FILE):
    df_occ_raw = pd.read_excel(OCCUPANCY_FILE)
    print(f"   Cargado: {len(df_occ_raw)} filas")
    print(f"   Columnas: {df_occ_raw.columns.tolist()}")

    # Detectar columna de fecha con prioridad: 'Date' exacto > 'DateTime' > tipo datetime64 con horas > nombre aproximado
    date_col = None
    for col in df_occ_raw.columns:
        if col == 'Date':
            date_col = col
            break
    if date_col is None:
        for col in df_occ_raw.columns:
            if col in ['DateTime', 'Datetime']:
                date_col = col
                break
    if date_col is None:
        for col in df_occ_raw.columns:
            if df_occ_raw[col].dtype == 'datetime64[ns]':
                if df_occ_raw[col].dt.hour.nunique() > 1:
                    date_col = col
                    break
    if date_col is None:
        for col in df_occ_raw.columns:
            col_lower = str(col).lower()
            if any(x in col_lower for x in ['date', 'fecha', 'hora', 'time', 'datetime']):
                date_col = col
                break
    if date_col is None:
        date_col = df_occ_raw.columns[0]

    print(f"   Columna de fecha detectada: '{date_col}'")

    # Convertir a datetime
    df_occ_raw['DateTime'] = pd.to_datetime(df_occ_raw[date_col], errors='coerce', dayfirst=True)
    df_occ_raw = df_occ_raw.dropna(subset=['DateTime'])
    print(f"   Fechas válidas: {len(df_occ_raw)}")
    print(f"   Rango: {df_occ_raw['DateTime'].min()} → {df_occ_raw['DateTime'].max()}")

    # Detectar columna de ocupación por nombre aproximado o tipo numérico
    occ_col = None
    for col in df_occ_raw.columns:
        col_lower = str(col).lower()
        if any(x in col_lower for x in ['occup', 'person', 'ocup', 'people', 'nº']):
            occ_col = col
            break
    if occ_col is None:
        for col in df_occ_raw.columns:
            if col != date_col and col != 'DateTime':
                if df_occ_raw[col].dtype in ['int64', 'float64']:
                    if 'li' not in str(col).lower() and 'ld' not in str(col).lower() and 'unnamed' not in str(col).lower():
                        occ_col = col
                        break
    if occ_col is None:
        occ_col = df_occ_raw.columns[-1]

    print(f"   Columna de ocupación detectada: '{occ_col}'")

    # Construir serie de ocupación indexada por fecha
    df_occ_raw['Occupancy'] = pd.to_numeric(df_occ_raw[occ_col], errors='coerce')
    df_occ = df_occ_raw.dropna(subset=['Occupancy'])[['DateTime', 'Occupancy']].copy()
    df_occ = df_occ.set_index('DateTime').sort_index()
    print(f"   Registros válidos de ocupación: {len(df_occ)}")
    print(f"   Valores únicos: {sorted(df_occ['Occupancy'].unique())[:20]}...")
    print(f"   Ocupación máxima: {df_occ['Occupancy'].max()}")
else:
    print(f"   ⚠️ Archivo no encontrado: {OCCUPANCY_FILE}")



CARGANDO DATOS DE OCUPACIÓN...
   Cargado: 82 filas
   Columnas: ['Unnamed: 0', 'Fecha', 'Hora', 'Date', 'LI (pasillo)', 'LD (jardín)', 'Occupancy']
   Columna de fecha detectada: 'Date'
   Fechas válidas: 82
   Rango: 2026-02-09 10:17:00 → 2026-03-18 11:15:00
   Columna de ocupación detectada: 'Occupancy'
   Registros válidos de ocupación: 78
   Valores únicos: [np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0), np.float64(8.0), np.float64(9.0), np.float64(11.0), np.float64(12.0), np.float64(14.0), np.float64(15.0), np.float64(16.0), np.float64(17.0), np.float64(18.0), np.float64(21.0), np.float64(22.0), np.float64(23.0)]...
   Ocupación máxima: 29.0


In [8]:
# ==========================================
# CARGAR DATOS DE SENSORES
# ==========================================
# Lee todos los archivos Excel semanales del aula 1.6 y extrae
# las variables interiores (temperatura, CO2, humedad) y exteriores
file_pattern = '1.6_data_*_week_*.xlsx'
all_files = sorted(glob.glob(file_pattern))

if not all_files:
    print(f"\nERROR: No se encontraron archivos con el patrón '{file_pattern}'")
    exit()

print(f"\nArchivos encontrados: {len(all_files)}")
print("\n" + "="*60)
print("CARGANDO DATOS DE SENSORES...")
print("="*60)

all_records = []

for file_path in all_files:
    fname = os.path.basename(file_path)
    print(f"   {fname}...", end=" ")

    df = pd.read_excel(file_path)
    metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']
    date_cols = [c for c in df.columns if c not in metadata_cols]
    timestamps = pd.to_datetime(date_cols, errors='coerce')

    def get_series(df, variable, location):
        """Extrae la serie temporal de una variable y ubicación concretas del pivote semanal."""
        mask = (df['Variable_Measure'] == variable) & (df['Sensor_Location'] == location)
        row = df[mask]
        if len(row) > 0:
            return row[date_cols].iloc[0].astype(float)
        return pd.Series(np.nan, index=date_cols)

    # Variables interiores del sensor del pasillo
    temp_corridor = get_series(df, 'Temperature', 'Corridor')
    co2_corridor  = get_series(df, 'CO2', 'Corridor')
    hum_corridor  = get_series(df, 'Humidity', 'Corridor')

    # Variables exteriores del archivo SIAR
    temp_outdoor  = get_series(df, 'Temperature_Outdoor', 'Outdoor')
    hum_outdoor   = get_series(df, 'Humidity_Outdoor', 'Outdoor')
    solar_outdoor = get_series(df, 'Solar_Radiation', 'Outdoor')

    # Construir DataFrame con todas las variables alineadas por timestamp
    df_mini = pd.DataFrame({
        'Temperature_Corridor': temp_corridor.values,
        'CO2_Corridor':         co2_corridor.values,
        'Humidity_Corridor':    hum_corridor.values,
        'Temperature_Outdoor':  temp_outdoor.values,
        'Humidity_Outdoor':     hum_outdoor.values,
        'Solar_Radiation':      solar_outdoor.values,
    }, index=timestamps)

    # Añadir ocupación buscando el registro más cercano en el tiempo (máx. 30 min)
    occ_count = 0
    if df_occ is not None:
        occupancy_values = []
        for ts in timestamps:
            if pd.isnull(ts):
                occupancy_values.append(np.nan)
                continue
            time_diffs = abs(df_occ.index - ts)
            min_diff = time_diffs.min()
            if min_diff <= pd.Timedelta('30min'):
                closest_idx = time_diffs.argmin()
                occupancy_values.append(df_occ.iloc[closest_idx]['Occupancy'])
            else:
                occupancy_values.append(np.nan)
        df_mini['Occupancy'] = occupancy_values
        occ_count = sum(1 for v in occupancy_values if not pd.isna(v))

    all_records.append(df_mini)
    print(f"OK ({len(df_mini)} registros, {occ_count} con ocupación)")

# Unir todos los datos semanales en un único DataFrame ordenado por fecha
print("\n   Concatenando todos los datos...")
df_all = pd.concat(all_records, ignore_index=False)
df_all = df_all.sort_index()
df_all['Month'] = df_all.index.month_name()

print(f"   Total registros: {len(df_all)}")
print(f"   Meses disponibles: {df_all['Month'].unique().tolist()}")

if 'Occupancy' in df_all.columns:
    occ_valid = df_all['Occupancy'].notna().sum()
    print(f"   Registros con ocupación: {occ_valid} / {len(df_all)}")
    if occ_valid > 0:
        print(f"   Valores de ocupación: {sorted(df_all['Occupancy'].dropna().unique())}")
        print(f"   Rango de ocupación: {df_all['Occupancy'].min():.0f} - {df_all['Occupancy'].max():.0f}")

# ==========================================
# CONFIGURACIÓN
# ==========================================
month_order = ['December', 'January', 'February', 'March']
unique_months = [m for m in month_order if m in df_all['Month'].unique()]
print(f"\n   Meses con datos: {unique_months}")

sensor_colors  = {'Corridor': '#2196F3'}
sensor_markers = {'Corridor': 'o'}

# ==========================================
# FUNCIÓN: Scatter interior (1 sensor)
# ==========================================
def create_scatter_1sensor(x_col, y_col, x_label, y_label, suptitle, filename):
    """Genera un scatter 2×2 (uno por mes) de dos variables interiores,
    con línea de tendencia y coeficiente R² en cada panel."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.01)

    axes_flat = axes.flatten()
    month_positions = {'December': 0, 'January': 1, 'February': 2, 'March': 3}
    any_data = False

    for month, pos in month_positions.items():
        ax = axes_flat[pos]

        if month not in unique_months:
            ax.set_visible(False)
            continue

        data_month = df_all[df_all['Month'] == month]

        if x_col not in data_month.columns or y_col not in data_month.columns:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, 'Sin datos', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        df_loc = data_month.dropna(subset=[x_col, y_col])

        if len(df_loc) < 3:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, f'Solo {len(df_loc)} puntos', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        any_data = True

        ax.scatter(df_loc[x_col], df_loc[y_col],
                   c=sensor_colors['Corridor'],
                   marker=sensor_markers['Corridor'],
                   alpha=0.5, s=15, edgecolors='none',
                   label='Corridor')

        # Línea de tendencia lineal con R²
        try:
            x_v = df_loc[x_col].values
            y_v = df_loc[y_col].values
            if len(x_v) > 2:
                z = np.polyfit(x_v, y_v, 1)
                p = np.poly1d(z)
                x_t = np.linspace(x_v.min(), x_v.max(), 100)
                ax.plot(x_t, p(x_t), color='red', linewidth=1.5,
                        linestyle='--', alpha=0.8)
                r2 = np.corrcoef(x_v, y_v)[0, 1] ** 2
                ax.text(0.05, 0.95, f'R² = {r2:.3f}',
                        transform=ax.transAxes, fontsize=9,
                        verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        except Exception:
            pass

        # Márgenes automáticos para evitar que los puntos queden cortados
        x_min, x_max = df_loc[x_col].min(), df_loc[x_col].max()
        x_margin = (x_max - x_min) * 0.1 if x_max > x_min else 5
        ax.set_xlim(x_min - x_margin, x_max + x_margin)

        y_min, y_max = df_loc[y_col].min(), df_loc[y_col].max()
        y_margin = (y_max - y_min) * 0.1 if y_max > y_min else 10
        ax.set_ylim(y_min - y_margin, y_max + y_margin)

        ax.set_title(f'{month}  |  n={len(df_loc)}', fontsize=11, fontweight='bold')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

    if not any_data:
        print(f"   OMITIDO: Sin datos para {suptitle}")
        plt.close()
        return

    plt.tight_layout()
    file_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(file_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   GUARDADO: {filename}")

# ==========================================
# FUNCIÓN: Scatter exterior vs interior
# ==========================================
def create_scatter_outdoor_1sensor(x_var, y_col, x_label, y_label, suptitle, filename):
    """Genera un scatter 2×2 (uno por mes) de una variable exterior frente
    a una variable interior, con línea de tendencia y R²."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle(suptitle, fontsize=15, fontweight='bold', y=1.01)

    axes_flat = axes.flatten()
    month_positions = {'December': 0, 'January': 1, 'February': 2, 'March': 3}
    any_data = False

    for month, pos in month_positions.items():
        ax = axes_flat[pos]

        if month not in unique_months:
            ax.set_visible(False)
            continue

        data_month = df_all[df_all['Month'] == month]

        if x_var not in data_month.columns or y_col not in data_month.columns:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, 'Sin datos', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        df_loc = data_month.dropna(subset=[x_var, y_col])

        if len(df_loc) < 3:
            ax.set_title(f'{month}', fontsize=12, fontweight='bold')
            ax.text(0.5, 0.5, f'Solo {len(df_loc)} puntos', transform=ax.transAxes,
                    ha='center', va='center', fontsize=11, color='gray')
            ax.set_xlabel(x_label, fontsize=10)
            ax.set_ylabel(y_label, fontsize=10)
            continue

        any_data = True

        ax.scatter(df_loc[x_var], df_loc[y_col],
                   c=sensor_colors['Corridor'],
                   marker=sensor_markers['Corridor'],
                   alpha=0.5, s=15, edgecolors='none',
                   label='Corridor')

        # Línea de tendencia lineal con R²
        try:
            x_v = df_loc[x_var].values
            y_v = df_loc[y_col].values
            if len(x_v) > 2:
                z = np.polyfit(x_v, y_v, 1)
                p = np.poly1d(z)
                x_t = np.linspace(x_v.min(), x_v.max(), 100)
                ax.plot(x_t, p(x_t), color='red', linewidth=1.5,
                        linestyle='--', alpha=0.8)
                r2 = np.corrcoef(x_v, y_v)[0, 1] ** 2
                ax.text(0.05, 0.95, f'R² = {r2:.3f}',
                        transform=ax.transAxes, fontsize=9,
                        verticalalignment='top',
                        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
        except Exception:
            pass

        # Márgenes automáticos
        x_min, x_max = df_loc[x_var].min(), df_loc[x_var].max()
        x_margin = (x_max - x_min) * 0.1 if x_max > x_min else 5
        ax.set_xlim(x_min - x_margin, x_max + x_margin)

        y_min, y_max = df_loc[y_col].min(), df_loc[y_col].max()
        y_margin = (y_max - y_min) * 0.1 if y_max > y_min else 10
        ax.set_ylim(y_min - y_margin, y_max + y_margin)

        ax.set_title(f'{month}  |  n={len(df_loc)}', fontsize=11, fontweight='bold')
        ax.set_xlabel(x_label, fontsize=10)
        ax.set_ylabel(y_label, fontsize=10)
        ax.grid(True, alpha=0.25, linestyle='--')
        ax.tick_params(labelsize=9)
        ax.legend(loc='upper right', fontsize=9, framealpha=0.9)

    if not any_data:
        print(f"   OMITIDO: Sin datos para {suptitle}")
        plt.close()
        return

    plt.tight_layout()
    file_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(file_path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"   GUARDADO: {filename}")

# ==========================================
# GENERAR GRÁFICOS — VARIABLES INTERIORES
# ==========================================
print("\n" + "="*60)
print("GENERANDO SCATTER PLOTS (sensor Corridor)")
print("="*60)

print("\n--- VARIABLES INTERIORES ---")

create_scatter_1sensor('Temperature_Corridor', 'CO2_Corridor',
    'Temperatura (°C)', 'CO₂ (ppm)',
    'CO₂ vs Temperatura — Aula 1.6',
    '01_CO2_vs_Temperature.png')

create_scatter_1sensor('CO2_Corridor', 'Temperature_Corridor',
    'CO₂ (ppm)', 'Temperatura (°C)',
    'Temperatura vs CO₂ — Aula 1.6',
    '02_Temperature_vs_CO2.png')

create_scatter_1sensor('CO2_Corridor', 'Humidity_Corridor',
    'CO₂ (ppm)', 'Humedad (%)',
    'Humedad vs CO₂ — Aula 1.6',
    '03_Humidity_vs_CO2.png')

create_scatter_1sensor('Humidity_Corridor', 'CO2_Corridor',
    'Humedad (%)', 'CO₂ (ppm)',
    'CO₂ vs Humedad — Aula 1.6',
    '04_CO2_vs_Humidity.png')

create_scatter_1sensor('Humidity_Corridor', 'Temperature_Corridor',
    'Humedad (%)', 'Temperatura (°C)',
    'Temperatura vs Humedad — Aula 1.6',
    '05_Temperature_vs_Humidity.png')

# CO₂ vs Ocupación — solo si hay suficientes datos de ocupación
if 'Occupancy' in df_all.columns and df_all['Occupancy'].notna().sum() > 3:
    create_scatter_1sensor('Occupancy', 'CO2_Corridor',
        'Ocupación (personas)', 'CO₂ (ppm)',
        'CO₂ vs Ocupación — Aula 1.6',
        '06_CO2_vs_Occupancy.png')
else:
    occ_count = df_all['Occupancy'].notna().sum() if 'Occupancy' in df_all.columns else 0
    print(f"   OMITIDO CO₂ vs Ocupación: solo {occ_count} registros de ocupación")

# ==========================================
# GENERAR GRÁFICOS — VARIABLES EXTERIORES
# ==========================================
print("\n--- VARIABLES EXTERIORES ---")

create_scatter_outdoor_1sensor('Temperature_Outdoor', 'Temperature_Corridor',
    'Temperatura exterior (°C)', 'Temperatura interior (°C)',
    'Temperatura interior vs exterior — Aula 1.6',
    '11_Indoor_vs_Outdoor_Temperature.png')

create_scatter_outdoor_1sensor('Humidity_Outdoor', 'Humidity_Corridor',
    'Humedad exterior (%)', 'Humedad interior (%)',
    'Humedad interior vs exterior — Aula 1.6',
    '12_Indoor_vs_Outdoor_Humidity.png')

create_scatter_outdoor_1sensor('Solar_Radiation', 'Temperature_Corridor',
    'Radiación solar (W/m²)', 'Temperatura interior (°C)',
    'Temperatura interior vs Radiación solar — Aula 1.6',
    '13_Temperature_vs_Solar_Radiation.png')

create_scatter_outdoor_1sensor('Temperature_Outdoor', 'CO2_Corridor',
    'Temperatura exterior (°C)', 'CO₂ interior (ppm)',
    'CO₂ interior vs Temperatura exterior — Aula 1.6',
    '14_CO2_vs_Outdoor_Temperature.png')

print("\n" + "="*60)
print("GRÁFICOS AULA 1.6 COMPLETADOS")
print("="*60)
print(f"\nArchivos guardados en '{OUTPUT_DIR}/'")


Archivos encontrados: 12

CARGANDO DATOS DE SENSORES...
   1.6_data_December_2025_week_4_days_22-31.xlsx... OK (824 registros, 0 con ocupación)
   1.6_data_February_2026_week_1_days_1-7.xlsx... OK (530 registros, 0 con ocupación)
   1.6_data_February_2026_week_2_days_8-14.xlsx... OK (540 registros, 58 con ocupación)
   1.6_data_February_2026_week_3_days_15-21.xlsx... OK (542 registros, 64 con ocupación)
   1.6_data_February_2026_week_4_days_22-28.xlsx... OK (4012 registros, 159 con ocupación)
   1.6_data_January_2026_week_1_days_1-7.xlsx... OK (530 registros, 0 con ocupación)
   1.6_data_January_2026_week_2_days_8-14.xlsx... OK (531 registros, 0 con ocupación)
   1.6_data_January_2026_week_3_days_15-21.xlsx... OK (530 registros, 0 con ocupación)
   1.6_data_January_2026_week_4_days_22-31.xlsx... OK (742 registros, 0 con ocupación)
   1.6_data_March_2026_week_1_days_1-7.xlsx... OK (5255 registros, 374 con ocupación)
   1.6_data_March_2026_week_2_days_8-14.xlsx... OK (5255 registros, 53